# 24: Positional Encoding - Teaching Position

## The Position Problem

**Problem**: Pure attention has no sense of order!
- "cat dog" and "dog cat" produce same attention patterns
- All positions processed in parallel (unlike RNN's sequential processing)
- No inherent way to know "this word came first"

**Solution**: **Positional encoding** - add position information to embeddings

### The Web Dev Analogy

Positional encoding is like **array indices or timestamps**:
- Without: `Set{"apple", "banana", "cherry"}` - no order
- With: `[{0: "apple"}, {1: "banana"}, {2: "cherry"}]` - ordered
- Transforms unordered set into ordered sequence

## What You'll Learn
- [ ] Explain why transformers need explicit positional information
- [ ] Implement sinusoidal positional encoding
- [ ] Visualize positional encoding patterns and their properties

## Connection to Previous Lessons

| What you learned | How it connects here |
|-----------------|---------------------|
| **Lesson 17**: RNN processes tokens *in order* (position is implicit) | Transformers process all tokens *in parallel* — so they have no built-in notion of position |
| **Lesson 23**: Self-attention treats input as a set | Without positional encoding, "dog bites man" = "man bites dog" again! |

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)

print("Ready to encode positions! 📍")

## 1. Why Attention Needs Position

In [ ]:
# Demonstrate position-invariance of pure attention
def simple_attention(query, keys, values):
    scores = torch.matmul(keys, query.unsqueeze(1)).squeeze()
    weights = torch.softmax(scores, dim=0)
    context = torch.matmul(weights, values)
    return context, weights

# Two sentences with same words, different order
embeddings_1 = torch.tensor([
    [1.0, 0.0],  # "dog"
    [0.0, 1.0],  # "cat"
])

embeddings_2 = torch.tensor([
    [0.0, 1.0],  # "cat"
    [1.0, 0.0],  # "dog"
])

query = torch.tensor([0.5, 0.5])

context_1, weights_1 = simple_attention(query, embeddings_1, embeddings_1)
context_2, weights_2 = simple_attention(query, embeddings_2, embeddings_2)

print("Sentence 1: dog cat")
print(f"  Attention weights: {weights_1.numpy()}")
print(f"  Context: {context_1.numpy()}")

print("\nSentence 2: cat dog")
print(f"  Attention weights: {weights_2.numpy()}")
print(f"  Context: {context_2.numpy()}")

print("\n❌ Different word order, but attention treats them the same!")
print("   We need positional information!")

`★ Insight ─────────────────────────────────────`

**Why RNNs don't need positional encoding:**
- Process sequentially (word 1, then 2, then 3...)
- Position is implicit in processing order
- Hidden state carries sequential information

**Why Transformers do:**
- Process all positions in parallel
- No inherent sequence information
- Must explicitly encode position

`─────────────────────────────────────────────────`

## 2. Simple Positional Encoding: Learned

In [ ]:
# Simplest approach: Learn a position embedding for each position
max_seq_len = 10
embedding_dim = 4

# Create learned positional embeddings
position_embeddings = nn.Embedding(max_seq_len, embedding_dim)

# Positions 0 to 9
positions = torch.arange(max_seq_len)
pos_encodings = position_embeddings(positions)

print(f"Learned positional embeddings:")
print(f"Shape: {pos_encodings.shape}")
print(f"\nFirst 3 positions:\n{pos_encodings[:3]}")

# Usage: Add to word embeddings
word_embeddings = torch.randn(max_seq_len, embedding_dim)
combined = word_embeddings + pos_encodings

print(f"\nWord embeddings + Positional encodings")
print(f"Result shape: {combined.shape}")

print("\n✓ Pros: Simple, learns from data")
print("✗ Cons: Fixed max length, doesn't generalize to longer sequences")

## 3. Sinusoidal Positional Encoding

The original Transformer paper uses **sinusoidal functions**:

```
PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
```

Where:
- `pos` = position in sequence
- `i` = dimension index
- `d_model` = embedding dimension

In [ ]:
def sinusoidal_positional_encoding(max_len, d_model):
    """
    Generate sinusoidal positional encodings.
    
    Args:
        max_len: Maximum sequence length
        d_model: Embedding dimension
    
    Returns:
        Positional encoding matrix (max_len, d_model)
    """
    pe = torch.zeros(max_len, d_model)
    
    # Position indices
    position = torch.arange(0, max_len).unsqueeze(1).float()
    
    # Dimension indices
    div_term = torch.exp(
        torch.arange(0, d_model, 2).float() * 
        (-np.log(10000.0) / d_model)
    )
    
    # Apply sin to even indices
    pe[:, 0::2] = torch.sin(position * div_term)
    
    # Apply cos to odd indices
    pe[:, 1::2] = torch.cos(position * div_term)
    
    return pe

# Generate encodings
max_len = 100
d_model = 64

pe = sinusoidal_positional_encoding(max_len, d_model)

print(f"Sinusoidal positional encoding:")
print(f"Shape: {pe.shape}")
print(f"\nFirst position encoding (first 8 dims):\n{pe[0, :8]}")
print(f"\nSecond position encoding (first 8 dims):\n{pe[1, :8]}")

## 4. Visualizing Sinusoidal Encodings

In [ ]:
# Visualize the positional encoding matrix
plt.figure(figsize=(12, 8))

plt.imshow(pe.numpy(), cmap='RdBu', aspect='auto')
plt.colorbar(label='Encoding Value')
plt.xlabel('Embedding Dimension', fontsize=12)
plt.ylabel('Position in Sequence', fontsize=12)
plt.title('Sinusoidal Positional Encoding', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Observations:")
print("   - Each position has a unique pattern")
print("   - Lower dimensions: Fast oscillations")
print("   - Higher dimensions: Slow oscillations")
print("   - Creates a unique 'fingerprint' for each position")

In [ ]:
# Visualize specific dimensions over positions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

dims_to_plot = [0, 4, 16, 32]

for idx, dim in enumerate(dims_to_plot):
    axes[idx].plot(pe[:50, dim].numpy(), linewidth=2)
    axes[idx].set_xlabel('Position', fontsize=11)
    axes[idx].set_ylabel('Encoding Value', fontsize=11)
    axes[idx].set_title(f'Dimension {dim}', fontsize=12, fontweight='bold')
    axes[idx].grid(True, alpha=0.3)
    axes[idx].axhline(y=0, color='black', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Different frequencies for different dimensions:")
print("   - Low dimensions: High frequency (fast oscillation)")
print("   - High dimensions: Low frequency (slow oscillation)")
print("   - Allows model to learn both local and global position info")

`★ Insight ─────────────────────────────────────`

**Why sinusoidal encoding is clever:**
1. **Deterministic**: No learning required
2. **Unique**: Each position has distinct encoding
3. **Relative position**: Can learn "k steps away" relationships
4. **Extrapolates**: Works for longer sequences than seen during training
5. **Multiple scales**: Different frequencies capture different ranges

Think: Like binary numbers but with continuous sine/cosine waves!

`─────────────────────────────────────────────────`

## 5. Relative Positions

In [ ]:
# Sinusoidal encoding has a cool property:
# PE(pos + k) can be represented as linear function of PE(pos)

# This allows the model to learn relative positions!

# Compute similarity between positions
pe_small = sinusoidal_positional_encoding(20, 16)

# Cosine similarity between positions
def cosine_similarity(a, b):
    return torch.dot(a, b) / (torch.norm(a) * torch.norm(b))

# Compare position 5 with all other positions
reference_pos = 5
similarities = []

for pos in range(20):
    sim = cosine_similarity(pe_small[reference_pos], pe_small[pos])
    similarities.append(sim.item())

plt.figure(figsize=(10, 6))
plt.plot(similarities, 'o-', linewidth=2, markersize=8)
plt.axvline(x=reference_pos, color='red', linestyle='--', label=f'Reference position ({reference_pos})')
plt.xlabel('Position', fontsize=12)
plt.ylabel('Cosine Similarity', fontsize=12)
plt.title(f'Similarity to Position {reference_pos}', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Nearby positions are more similar!")
print("   Model can learn to pay more attention to nearby words.")

## 6. Using Positional Encoding in Practice

In [ ]:
class PositionalEncoding(nn.Module):
    """Positional encoding layer for Transformers."""
    
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * 
            (-np.log(10000.0) / d_model)
        )
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Add batch dimension
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        
        # Register as buffer (not a parameter, but should be saved with model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """
        Args:
            x: Input embeddings (batch, seq_len, d_model)
        
        Returns:
            x + positional encoding (batch, seq_len, d_model)
        """
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len, :]
        return self.dropout(x)

# Test it
d_model = 64
batch_size = 2
seq_len = 10

pos_encoder = PositionalEncoding(d_model)

# Simulate word embeddings
word_embeddings = torch.randn(batch_size, seq_len, d_model)

# Add positional encoding
output = pos_encoder(word_embeddings)

print(f"Input shape: {word_embeddings.shape}")
print(f"Output shape: {output.shape}")
print("\n✅ Same shape! Positional info added to embeddings.")

## 7. Learned vs Sinusoidal: Comparison

In [ ]:
print("Positional Encoding Comparison:")
print("=" * 70)

print("\n1. Learned Embeddings:")
print("   ✓ Simple to implement (just nn.Embedding)")
print("   ✓ Can adapt to specific task")
print("   ✗ Fixed maximum length")
print("   ✗ Doesn't generalize to longer sequences")
print("   ✗ Requires learning (more parameters)")
print("   → Used by: BERT, GPT-2")

print("\n2. Sinusoidal Encoding:")
print("   ✓ No parameters to learn")
print("   ✓ Generalizes to any length")
print("   ✓ Encodes relative positions naturally")
print("   ✗ Fixed pattern (not task-adaptive)")
print("   ✗ Slightly more complex")
print("   → Used by: Original Transformer")

print("\n3. Relative Position Encodings:")
print("   ✓ Directly encodes relative distances")
print("   ✓ More flexible for long sequences")
print("   ✓ Used in modern architectures")
print("   ✗ More complex implementation")
print("   → Used by: Transformer-XL, T5")

print("\n💡 In practice: Both work well! Choice depends on use case.")

## 8. Ablation: What Happens Without Position?

In [ ]:
# Demonstrate importance of positional encoding
print("Experiment: Position Encoding Ablation")
print("=" * 70)

# Simulate two sentences with same words, different order
# "not good movie" vs "good movie not"

vocab = {"not": 0, "good": 1, "movie": 2}
d_model = 8

# Word embeddings (learned)
embedding = nn.Embedding(len(vocab), d_model)

sentence1 = torch.tensor([0, 1, 2])  # "not good movie"
sentence2 = torch.tensor([1, 2, 0])  # "good movie not"

# Without positional encoding
embed1_no_pos = embedding(sentence1)
embed2_no_pos = embedding(sentence2)

# With positional encoding
pos_enc = PositionalEncoding(d_model, max_len=3, dropout=0)
embed1_with_pos = pos_enc(embed1_no_pos.unsqueeze(0)).squeeze(0)
embed2_with_pos = pos_enc(embed2_no_pos.unsqueeze(0)).squeeze(0)

# Compare
def sentence_similarity(emb1, emb2):
    # Average pooling
    avg1 = emb1.mean(dim=0)
    avg2 = emb2.mean(dim=0)
    return F.cosine_similarity(avg1, avg2, dim=0).item()

sim_no_pos = sentence_similarity(embed1_no_pos, embed2_no_pos)
sim_with_pos = sentence_similarity(embed1_with_pos, embed2_with_pos)

print(f"\nSentence 1: 'not good movie'")
print(f"Sentence 2: 'good movie not'")
print(f"\nSimilarity WITHOUT positional encoding: {sim_no_pos:.4f}")
print(f"Similarity WITH positional encoding:    {sim_with_pos:.4f}")

print("\n✅ With position: Different word orders are distinguished!")
print("❌ Without position: Same words = same representation (bag-of-words)")

## 📝 Check Your Understanding

1. Why do Transformers need positional encoding?
2. What are the advantages of sinusoidal encoding over learned?
3. How do different frequencies help encode position?
4. What would happen without positional encoding?
5. How does positional encoding help with relative positions?

In [ ]:
# --- Exercise 1: Positional Encoding Values ---
# Compute PE values for position 0.
# PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
# PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
# At position 0: sin(0) = 0, cos(0) = 1

# YOUR CODE HERE:
pe_0_0 = None  # PE(position=0, dimension=0) = sin(0)
pe_0_1 = None  # PE(position=0, dimension=1) = cos(0)

# --- Check ---
assert pe_0_0 is not None and pe_0_1 is not None, "Compute both values!"
assert abs(pe_0_0 - 0.0) < 0.001, f"sin(0) = 0, got {pe_0_0}"
assert abs(pe_0_1 - 1.0) < 0.001, f"cos(0) = 1, got {pe_0_1}"
print("Exercise 1 passed! ✓")

# --- Quick Check: Why Positional Encoding? ---
# Why can't self-attention distinguish "dog bites man" from "man bites dog"
# without positional encoding?
# a) Self-attention can't process more than 2 tokens
# b) Self-attention treats input as a SET (order-invariant) — same tokens produce same attention
# c) Self-attention requires RNN preprocessing first
# d) Self-attention only works with numbers, not words

your_answer = None  # Put 'a', 'b', 'c', or 'd'

# --- Check ---
assert your_answer is not None, "Pick an answer!"
assert your_answer == 'b', "Self-attention computes the same Q,K,V regardless of token order — it's permutation-invariant!"
print("Exercise 2 passed! ✓")

print("\n🎉 All exercises passed!")

## 🎯 Summary

**Why positional encoding:**
- Transformers process all positions in parallel
- No inherent sequence order (unlike RNNs)
- Must explicitly inject position information

**Two main approaches:**
1. **Learned**: `nn.Embedding` for positions
   - Simple, adaptive, but fixed length
2. **Sinusoidal**: `sin/cos` functions
   - No learning, generalizes, encodes relative position

**Sinusoidal formula:**
```
PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
```

**Key properties:**
- Each position gets unique encoding
- Different frequencies for different dimensions
- Allows learning relative positions
- Added to word embeddings

**Implementation:**
- Compute once, reuse for all sequences
- Add to embeddings: `x = word_emb + pos_enc`
- Register as buffer (not parameter)

**Next up**: Full Transformer architecture! →